<a target="_blank" href="https://colab.research.google.com/github/FranQuant/the_ai_engineer_capstones/blob/main/capstones/week02_backprop/04_pytorch_nn_module.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# 04 — PyTorch nn.Module + DataLoader (1-Hidden-Layer MLP)

Stage 4 of the Week-02 capstone: wrap the two-layer XOR MLP in `nn.Module`, preserving the same mathematical model and data-generating process as Notebooks 01–03, and train with mini-batch SGD using PyTorch's standard `MSELoss`.

## 1. Imports & Deterministic Seeds

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

# ------------------------------
# Deterministic seeds (match 01/02/03)
# ------------------------------
SEED = 42
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)

def set_seed(seed=42):
    global rng
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)

set_seed(SEED)
print("Seeds set to", SEED)

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────
SEED       = 42
D          = 2      # input dimension
H          = 4      # hidden units
LR         = 0.1    # SGD learning rate
EPOCHS     = 200
BATCH_SIZE = 16
N_SAMPLES  = 500
# ────────────────────────────────────────────────────────────────────
torch.manual_seed(SEED)
np.random.seed(SEED)

## 2. Synthetic Dataset (XOR logic)

Matches Notebooks 01–03:

$$x \sim \mathrm{Uniform}([-1,\,1]^2), \quad y = \mathbf{1}[x_1 \cdot x_2 < 0]$$

An 80/20 stratified train–validation split is applied before creating `DataLoader`s.

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

def generate_toy_data(n_samples=N_SAMPLES):
    X = rng.uniform(-1, 1, size=(n_samples, 2)).astype(np.float32)
    y = (X[:, 0] * X[:, 1] < 0).astype(np.float32)
    return X, y

X_np, y_np = generate_toy_data()

X_train_np, X_val_np, y_train_np, y_val_np = train_test_split(
    X_np, y_np, test_size=0.2, random_state=SEED, stratify=y_np)

# All-data tensors (used for final accuracy computation)
X_all = torch.tensor(X_np, dtype=torch.float32)
y_all = torch.tensor(y_np, dtype=torch.float32)

train_ds = TensorDataset(
    torch.tensor(X_train_np, dtype=torch.float32),
    torch.tensor(y_train_np, dtype=torch.float32))
val_ds = TensorDataset(
    torch.tensor(X_val_np, dtype=torch.float32),
    torch.tensor(y_val_np, dtype=torch.float32))

gen = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, generator=gen)
val_loader   = DataLoader(val_ds,   batch_size=len(val_ds))

print(f"Total: {len(X_np)} | Train: {len(train_ds)} | Val: {len(val_ds)}")
print(f"Train batches/epoch: {len(train_loader)}")
print(f"X_all shape: {X_all.shape}  y_all shape: {y_all.shape}")

## 3. nn.Module Parameters + Forward

Manual parameters inside an `nn.Module`, mirroring $W_1, b_1, W_2, b_2$ from Notebooks 01–03:
- $W_1: (h, d)$, $b_1: (h,)$
- $W_2: (1, h)$, $b_2: (1,)$
- Gaussian $\mathcal{N}(0, 0.1)$ for weights, zeros for biases.

`last_h1` is stored after every forward pass for ReLU activity tracking.

In [ ]:
class TwoLayerXOR(torch.nn.Module):
    def __init__(self, d=2, h=4, out=1):
        super().__init__()
        W1 = torch.tensor(rng.normal(0.0, 0.1, size=(h, d)), dtype=torch.float32)
        b1 = torch.zeros(h, dtype=torch.float32)
        W2 = torch.tensor(rng.normal(0.0, 0.1, size=(out, h)), dtype=torch.float32)
        b2 = torch.zeros(out, dtype=torch.float32)
        self.W1 = torch.nn.Parameter(W1)
        self.b1 = torch.nn.Parameter(b1)
        self.W2 = torch.nn.Parameter(W2)
        self.b2 = torch.nn.Parameter(b2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        a1 = x @ self.W1.T + self.b1   # (batch, h)
        h1 = torch.relu(a1)            # (batch, h)
        self.last_h1 = h1
        f = h1 @ self.W2.T + self.b2   # (batch, 1)
        return f.squeeze(-1)           # (batch,)


model = TwoLayerXOR(d=D, h=H)
print(model)
print({name: tuple(p.shape) for name, p in model.named_parameters()})

with torch.no_grad():
    f0 = model(X_all[:4])
    print("Forward on 4 samples ->", f0.shape)

## 4. DataLoader (mini-batching)

Train and val `DataLoader`s are created in Section 2 alongside the dataset split.
This section confirms their configuration.

In [ ]:
# DataLoaders created in Section 2
print(f"Train batches/epoch: {len(train_loader)}")
print(f"Val batches/epoch:   {len(val_loader)}")
print(f"Batch size: {BATCH_SIZE}")

## 5. Loss Function & Optimizer (SGD)

Use mean-squared error (same objective, different scaling).

**Note:** Unlike Notebooks 01–03, which use  
$
\frac{1}{2}(f - y)^2
$
per sample, `torch.nn.MSELoss(reduction="mean")` computes the batch mean of  
$
(f - y)^2.
$

This rescales gradients by a constant factor and does not change the optimizer's fixed points.

In [ ]:
loss_fn = torch.nn.MSELoss(reduction="mean")
optimizer = torch.optim.SGD(model.parameters(), lr=LR)

print("optimizer:", optimizer)

## 6. Training Loop (mini-batch SGD + gradient norms + val loss)

Logs mean train and val loss per epoch, gradient $\ell_2$ norms per mini-batch, and the
fraction of active ReLU units per mini-batch.

In [ ]:
loss_history      = []
grad_norm_history = []
val_loss_hist     = []
relu_activity_hist = []
final_loss = None

model.train()
for epoch in range(1, EPOCHS + 1):
    epoch_loss = 0.0
    num_batches = 0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        preds = model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()

        # Gradient norm
        total_norm_sq = 0.0
        for p in model.parameters():
            if p.grad is not None:
                total_norm_sq += p.grad.norm().item() ** 2
        grad_norm = total_norm_sq ** 0.5
        grad_norm_history.append(grad_norm)

        # ReLU activity: fraction of units active in this batch
        relu_frac = (model.last_h1 > 0).float().mean().item()
        relu_activity_hist.append(relu_frac)

        optimizer.step()

        epoch_loss += loss.item()
        num_batches += 1

    mean_loss = epoch_loss / num_batches
    loss_history.append(mean_loss)
    final_loss = mean_loss

    # Validation loss (full val set in one batch)
    model.eval()
    with torch.no_grad():
        xv, yv = next(iter(val_loader))
        val_preds = model(xv)
        v_loss = loss_fn(val_preds, yv).item()
    val_loss_hist.append(v_loss)
    model.train()

    if epoch % 20 == 0:
        print(f"epoch {epoch:03d} | train_loss {mean_loss:.4f} | val_loss {v_loss:.4f} | grad-norm {grad_norm:.4f}")

print("Training done.")

## 7. Diagnostics

Three-panel figure: train/val loss curves, gradient norm trace, and fraction of active ReLU units per batch.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))

ax[0].plot(loss_history, label="train loss")
ax[0].plot(val_loss_hist, label="val loss")
ax[0].set_title("Training & Validation Loss")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("Loss")
ax[0].legend()
ax[0].grid(True)

ax[1].plot(grad_norm_history)
ax[1].set_title("Gradient Norms")
ax[1].set_xlabel("Batch step")
ax[1].set_ylabel("Grad norm (L2)")
ax[1].grid(True)

ax[2].plot(relu_activity_hist)
ax[2].set_title("Fraction of active ReLUs")
ax[2].set_xlabel("Iteration")
ax[2].set_ylabel("Active fraction")
ax[2].grid(True)

plt.tight_layout()

## 4b. Equivalent with nn.Sequential

In [ ]:
import torch.nn as nn

seq_model = nn.Sequential(
    nn.Linear(D, H),
    nn.ReLU(),
    nn.Linear(H, 1),
)
# Copy weights from TwoLayerXOR to confirm numerical parity
with torch.no_grad():
    seq_model[0].weight.copy_(model.W1)
    seq_model[0].bias.copy_(model.b1)
    seq_model[2].weight.copy_(model.W2)
    seq_model[2].bias.copy_(model.b2)
x_test = torch.tensor([[1., -1.]])
diff = abs(model(x_test).item() - seq_model(x_test).item())
print(f"TwoLayerXOR vs nn.Sequential output diff: {diff:.2e}")
assert diff < 1e-5, "Parity check failed"

In [ ]:
torch.save(model.state_dict(), "two_layer_xor.pt")
# Verify round-trip
model_loaded = TwoLayerXOR(D, H)
model_loaded.load_state_dict(torch.load("two_layer_xor.pt", weights_only=True))
model_loaded.eval()
assert abs(model(x_test).item() -
           model_loaded(x_test).item()) < 1e-6, "Checkpoint round-trip failed"
print("Checkpoint saved and verified.")

## 8. Accuracy on Final Model

Threshold the scalar output at 0.5 to compute XOR classification accuracy.

In [ ]:
model.eval()
with torch.no_grad():
    preds_full = model(X_all)
    y_hat = (preds_full >= 0.5).float()
    accuracy = (y_hat == y_all).float().mean().item()

print("Final accuracy:", round(accuracy, 4))
print("Sample predictions:", preds_full[:5].tolist())

## 9. Final Metrics Summary (loss & accuracy)

In [ ]:
# ============================================
# 9. Final Metrics Summary (loss & accuracy)
# ============================================

print(f"Final training loss: {final_loss:.4f}")
print(f"Final training accuracy: {accuracy:.4f}")

## 10. Gradient Norm Trace

Inspect the recorded gradient norms from the last batch of each epoch.

In [ ]:
print("Gradient norms (first 5):", grad_norm_history[:5])
print("Gradient norms (last 5):", grad_norm_history[-5:])

plt.figure(figsize=(6, 3.5))
plt.plot(grad_norm_history)
plt.xlabel("Batch step")
plt.ylabel("Grad norm (L2)")
plt.title("Gradient Norms over Training")
plt.grid(True);

## Final Notes

- The data-generating process, parameter shapes, initialization scheme, forward equations, and loss definition are consistent with Notebooks 01–03. Numerical values are not expected to be bitwise identical across notebooks, as random number generators are consumed independently and training now uses mini-batch SGD.
- The `nn.Module` mirrors the manual parameters ($W_1, b_1, W_2, b_2$) and uses the same ReLU hidden layer.
- **Val split (FIX 3)**: 20 % of data (100 samples) is held out as a stratified validation set; train and val loss curves are plotted together in the diagnostics figure.
- **ReLU activity (FIX 2)**: `model.last_h1` is recorded inside `forward`; the fraction of active units is tracked per mini-batch and shown in subplot 3 of the diagnostics figure.
- **nn.Sequential parity (FIX 1)**: trained weights are copied into an `nn.Sequential` equivalent and the output difference is asserted to be < 1e-5.
- **Checkpoint (FIX 4)**: model state dict is saved to `two_layer_xor.pt`; a round-trip `load_state_dict` is verified with < 1e-6 output tolerance.
- Mini-batch SGD converges quickly; loss and gradient norms shrink steadily. Final accuracy approaches 1.0, confirming the learned XOR mapping.